# **Task 1: Conceptual Understanding**

**1. What is the difference between "Love" and "love" in NLP?**\
**Ans:** In raw NLP, computers treat "Love" and "love" as two completely different entities because their ASCII/Unicode values are different. "Love" (capitalized) might be recognized as a proper noun or the start of a sentence, while "love" (lowercase) is treated as a general verb or noun. If we don't perform "Lowercasing" during preprocessing, the model will split the frequency count for the same word, making the data more sparse and less efficient.

**2. What happens if stopwords are not removed?**\
**Ans:** If stopwords (like "is", "the", "and") are not removed, they often dominate the frequency distribution because they appear so often. This can "drown out" the meaningful keywords that actually define the context of the text. In models like Word2Vec or TF-IDF, keeping stopwords can increase the computational noise and lead to less accurate results because the model might think these common words are more important than they actually are.

**3. Provide two real-world scenarios where removing stopwords can be harmful.\
Ans:**
*   **Sentiment Analysis:** In the sentence "I am not happy," the word "not" is a stopword. If you remove it, the sentence becomes "I happy," which completely flips the sentiment from negative to positive.
*   **Search Engines / Intent Recognition:** For a query like "To be or not to be," every single word is technically a stopword. Removing them would leave the search engine with zero input, losing the entire context of the famous quote.

**4. Explain the difference between stemming and lemmatization.\
Ans:**

*   **Stemming:** This is a "crude" process that chops off the ends of words to find the root. It often follows fixed rules, which can lead to non-dictionary words. For example, "studies" and "studying" might both become "studi". It is fast but less accurate.
*   **Lemmatization:** This is a more sophisticated process that uses a dictionary (morphological analysis) to return the word to its actual base form, known as a "Lemma." For example, "better" would be lemmatized to "good." It is more computationally expensive but highly accurate.



# **Task 2:Build Advanced Preprocessing Function**

In [5]:
import re

def preprocess_text(text):

    if not isinstance(text, str):
        return ""

    # 1. Remove numbers
    # Matches any digit and replaces it with nothing
    text = re.sub(r'\d+', '', text)

    # 2. Remove extra spaces
    # Replaces multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text).strip()

    # 3. Handle repeated characters
    # Reduces characters appearing 3 or more times to just 2 (e.g., "soooo" -> "so")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 4. Convert text to lowercase
    # Ensures uniform analysis by ignoring capitalization
    text = text.lower()

    # 5. Remove very short tokens (Length ≤ 2)
    # Exception: keep meaningful words like "no" and "not"
    meaningful_exceptions = {'no', 'not'}
    tokens = text.split()
    tokens = [t for t in tokens if len(t) > 2 or t in meaningful_exceptions]
    text = " ".join(tokens)

    # 6. Remove URLs and email-like patterns
    # Matches web protocols (http/https), www, and common email structures
    text = re.sub(r'https?://\S+|www\.\S+|\S+@\S+', '', text)

    # Final cleanup of punctuation (optional but recommended for clean tokens)
    text = re.sub(r'[^\w\s]', '', text).strip()

    return text

# --- TEST SUITE (Verifying based on your examples) ---
print(f"1. Numbers:          {preprocess_text('I have 2 dogs')}")
print(f"2. Extra Spaces:     {preprocess_text('This   is     good')}")
print(f"3. Repeated Chars:   {preprocess_text('soooo goooood!!!')}")
print(f"4. Lowercase:        {preprocess_text('WOW!!! This is GREAT!!!')}")
print(f"5. Short/Exceptions: {preprocess_text('he is no not happy')}")
print(f"6. Web Patterns:     {preprocess_text('Visit http://example.com now')}")

1. Numbers:          have dogs
2. Extra Spaces:     this good
3. Repeated Chars:   soo good
4. Lowercase:        wow this great
5. Short/Exceptions: no not happy
6. Web Patterns:     visit  now


# **Task 3: Stress Testing**

In [6]:
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("--- NLP Preprocessing Stress Test Results ---\n")

# Process and display each sentence in the required format
for i, text in enumerate(sample_inputs, 1):
    cleaned_sentence = preprocess_text(text)
    cleaned_tokens = cleaned_sentence.split()

    print(f"Sentence {i}:")
    print(f"Original Text:    {text}")
    print(f"Cleaned Tokens:   {cleaned_tokens}")
    print(f"Cleaned Sentence: {cleaned_sentence}")
    print("-" * 50)

--- NLP Preprocessing Stress Test Results ---

Sentence 1:
Original Text:    Get 100% FREE access now!!!
Cleaned Tokens:   ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
--------------------------------------------------
Sentence 2:
Original Text:    I absolutely looooved this product 😍😍
Cleaned Tokens:   ['absolutely', 'looved', 'this', 'product']
Cleaned Sentence: absolutely looved this product
--------------------------------------------------
Sentence 3:
Original Text:    Worst service ever... 0/10
Cleaned Tokens:   ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
--------------------------------------------------
Sentence 4:
Original Text:    Call me at 9876543210
Cleaned Tokens:   ['call']
Cleaned Sentence: call
--------------------------------------------------
Sentence 5:
Original Text:    This is THE best course!!!
Cleaned Tokens:   ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
--------------------------------

# **Task 4: Token Analytics**

In [9]:
print("--- Task 4: Token Analytics Report ---\n")

analytics_results = []

for i, text in enumerate(sample_inputs, 1):
    cleaned_sentence = preprocess_text(text)
    tokens = cleaned_sentence.split()

    # Compute Metrics
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_token_length = sum(len(t) for t in tokens) / total_tokens if total_tokens > 0 else 0

    # Calculate "Noise" (Difference in length between original and cleaned)
    noise_removed = len(text) - len(cleaned_sentence)

    analytics_results.append({
        'id': i,
        'original': text,
        'tokens': total_tokens,
        'unique': unique_tokens,
        'avg_len': round(avg_token_length, 2),
        'noise_score': noise_removed
    })

    print(f"Sentence {i}:")
    print(f"  Total Tokens:  {total_tokens}")
    print(f"  Unique Tokens: {unique_tokens}")
    print(f"  Avg Token Len: {round(avg_token_length, 2)}")
    print("-" * 30)

# --- Analysis Questions ---

# 1. Most Noise (determined by largest reduction in string length)
most_noise = max(analytics_results, key=lambda x: x['noise_score'])

# 2. Most Meaningful (determined by highest number of total tokens remaining)
most_meaningful = max(analytics_results, key=lambda x: x['tokens'])

print("\n--- Final Analysis ---")
print(f"Q: Which sentence had the most noise?")
print(f"A: Sentence {most_noise['id']} ('{most_noise['original']}') had the most noise due to URLs or long number strings.")

print(f"\nQ: Which sentence retained the most meaningful tokens?")
print(f"A: Sentence {most_meaningful['id']} ('{most_meaningful['original']}') retained {most_meaningful['tokens']} tokens.")

--- Task 4: Token Analytics Report ---

Sentence 1:
  Total Tokens:  4
  Unique Tokens: 4
  Avg Token Len: 4.0
------------------------------
Sentence 2:
  Total Tokens:  4
  Unique Tokens: 4
  Avg Token Len: 6.75
------------------------------
Sentence 3:
  Total Tokens:  3
  Unique Tokens: 3
  Avg Token Len: 5.33
------------------------------
Sentence 4:
  Total Tokens:  1
  Unique Tokens: 1
  Avg Token Len: 4.0
------------------------------
Sentence 5:
  Total Tokens:  4
  Unique Tokens: 4
  Avg Token Len: 4.25
------------------------------
Sentence 6:
  Total Tokens:  2
  Unique Tokens: 2
  Avg Token Len: 4.0
------------------------------
Sentence 7:
  Total Tokens:  3
  Unique Tokens: 3
  Avg Token Len: 3.67
------------------------------
Sentence 8:
  Total Tokens:  1
  Unique Tokens: 1
  Avg Token Len: 3.0
------------------------------
Sentence 9:
  Total Tokens:  4
  Unique Tokens: 4
  Avg Token Len: 4.5
------------------------------
Sentence 10:
  Total Tokens:  4
  Uniq

# **Task 5: Frequency Analysis**

In [10]:
from collections import Counter

# 1. Combine all tokens from all 10 sentences into one large list
all_tokens = []
for text in sample_inputs:
    cleaned_sentence = preprocess_text(text)
    tokens = cleaned_sentence.split()
    all_tokens.extend(tokens)

# 2. Initialize the Counter to track frequencies
word_freq = Counter(all_tokens)

# 3. Identify Top 10 most frequent words
top_10 = word_freq.most_common(10)

# 4. Identify Top 5 least frequent words
# .most_common() without arguments returns all; we take the last 5
least_5 = word_freq.most_common()[:-6:-1]

print("--- Task 5: Frequency Analysis Report ---\n")

print("Top 10 Most Frequent Words:")
for word, count in top_10:
    print(f"  - '{word}': {count}")

print("\nTop 5 Least Frequent Words:")
for word, count in least_5:
    print(f"  - '{word}': {count}")

# Bonus: Simple visualization of corpus diversity
print(f"\nCorpus Summary:")
print(f"  - Total Words Processed: {len(all_tokens)}")
print(f"  - Unique Vocabulary Size: {len(word_freq)}")

--- Task 5: Frequency Analysis Report ---

Top 10 Most Frequent Words:
  - 'this': 4
  - 'now': 3
  - 'get': 1
  - 'free': 1
  - 'access': 1
  - 'absolutely': 1
  - 'looved': 1
  - 'product': 1
  - 'worst': 1
  - 'service': 1

Top 5 Least Frequent Words:
  - 'with': 1
  - 'happy': 1
  - 'not': 1
  - 'offer': 1
  - 'limited': 1

Corpus Summary:
  - Total Words Processed: 30
  - Unique Vocabulary Size: 25


# **Task 6: Build Full Pipeline**

In [13]:
def full_pipeline(text_list):
    clean_sentences = []
    all_tokens = []

    for text in text_list:
        # Use the logic from Task 2 to clean each sentence
        cleaned = preprocess_text(text)
        clean_sentences.append(cleaned)

        # Split into tokens and add to the master list
        tokens = cleaned.split()
        all_tokens.extend(tokens)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

# --- Execution and Full Display ---
result = full_pipeline(sample_inputs)

print("--- Task 6: Full Pipeline (All 10 Sentences) ---\n")

# Display every cleaned sentence to verify the full list
for i, sentence in enumerate(result['clean_sentences'], 1):
    print(f"Sentence {i}: {sentence}")

print("\n" + "="*30)
print(f"TOTAL TOKENS IN CORPUS: {len(result['tokens'])}")
print("="*30)

# Print the final dictionary structure as requested
# print(result)  # Uncomment this line if you need to see the raw dictionary

--- Task 6: Full Pipeline (All 10 Sentences) ---

Sentence 1: get free access now
Sentence 2: absolutely looved this product
Sentence 3: worst service ever
Sentence 4: call
Sentence 5: this the best course
Sentence 6: visit  now
Sentence 7: noo this baad
Sentence 8: got
Sentence 9: win now limited offer
Sentence 10: not happy with this

TOTAL TOKENS IN CORPUS: 30


# **Task 7: Error Handling**

In [15]:
edge_cases = {
    "Empty String": "",
    "Only Emojis": "😍😍😍😍",
    "Only Numbers": "1234567890",
    "Spaces Only": "     ",
    "None Value": None
}

print("--- Task 7: Error Handling Results ---\n")

for case_name, test_input in edge_cases.items():
    try:
        cleaned = preprocess_text(test_input)

        # Determine status
        status = "Handled" if cleaned == "" else f"✨ Result: '{cleaned}'"

        print(f"Test Case: {case_name}")
        print(f"  Input:  '{test_input}'")
        print(f"  Output: '{cleaned}'")
        print(f"  Status: {status}")
        print("-" * 40)

    except Exception as e:
        print(f"Error in {case_name}: {e}")

print("\n All Edge Cases Passed! The Preprocessing Engine is robust.")

--- Task 7: Error Handling Results ---

Test Case: Empty String
  Input:  ''
  Output: ''
  Status: Handled
----------------------------------------
Test Case: Only Emojis
  Input:  '😍😍😍😍'
  Output: ''
  Status: Handled
----------------------------------------
Test Case: Only Numbers
  Input:  '1234567890'
  Output: ''
  Status: Handled
----------------------------------------
Test Case: Spaces Only
  Input:  '     '
  Output: ''
  Status: Handled
----------------------------------------
Test Case: None Value
  Input:  'None'
  Output: ''
  Status: Handled
----------------------------------------

 All Edge Cases Passed! The Preprocessing Engine is robust.
